In [ ]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

### Etapa 1: Importando e verificando arquivos

Nesta etapa do collab, estamos apenas conectando ao google drive e extraindo o zip que contém o dataset. A ideia é apenas verificar se os 2mil Parquat estão salvos corretamente

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import zipfile
from pathlib import Path

zip_path = Path('/content/drive/MyDrive/ProjetoDados/dataset.zip')
extract_path = Path('/content/dataset_extraido')

print(f"Extraindo {zip_path}... ")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extração concluída!")
# Listar arquivos parquet no diretório extraído para verificar
arquivos_parquet = sorted(extract_path.rglob('*.parquet'))
print(f"Arquivos Parquet encontrados após extração: {len(arquivos_parquet):,}")
if arquivos_parquet:
    print("\nPrimeiros arquivos Parquet encontrados:")
    for arquivo in arquivos_parquet[:10]:
        print(arquivo.relative_to(extract_path))

Extraindo /content/drive/MyDrive/ProjetoDados/dataset.zip... 
Extração concluída!
Arquivos Parquet encontrados após extração: 2,051

Primeiros arquivos Parquet encontrados:
-taverna_da_resenha_games_br-___-chat-.parquet
-taverna_da_resenha_games_br-___ajuda-em-jogos.parquet
-taverna_da_resenha_games_br-___akinator.parquet
-taverna_da_resenha_games_br-___artes.parquet
-taverna_da_resenha_games_br-___conversa-1.parquet
-taverna_da_resenha_games_br-___conversa-2.parquet
-taverna_da_resenha_games_br-___conversa-geral.parquet
-taverna_da_resenha_games_br-___cor-do-nome.parquet
-taverna_da_resenha_games_br-___futcord.parquet
-taverna_da_resenha_games_br-___jogos-1.parquet


### Etapa 2: Inspeção dos arquivos

O esquema identificado nesse arquivo será utilizado como referência para verificar a consistência estrutural dos demais arquivos Parquet. Essa verificação permitirá identificar diferenças na quantidade, nos nomes, na ordem e, posteriormente, nos tipos das colunas.

In [ ]:
import pyarrow.parquet as pq
from pathlib import Path
from collections import Counter

pasta = Path("/content/dataset_extraido")
arquivos = sorted(pasta.rglob("*.parquet"))

print(f"Total de arquivos: {len(arquivos):,}")

# Contabiliza os esquemas de colunas encontrados
esquemas = Counter()
erros = []

for arquivo in arquivos:
    try:
        schema = pq.read_schema(arquivo)

        # Considera os nomes e a ordem das colunas
        esquema = tuple(schema.names)
        esquemas[esquema] += 1

    except Exception as erro:
        erros.append({
            "arquivo": str(arquivo),
            "erro": str(erro)
        })

print(f"\nQuantidade de esquemas diferentes: {len(esquemas)}")

for i, (esquema, quantidade) in enumerate(esquemas.items(), start=1):
    print(f"\nEsquema {i}: {quantidade:,} arquivo(s)")
    print(esquema)

print(f"\nArquivos com erro de leitura: {len(erros)}")

Total de arquivos: 2,051

Quantidade de esquemas diferentes: 1

Esquema 1: 2,051 arquivo(s)
('type', 'content', 'mentions', 'mention_roles', 'attachments', 'embeds', 'timestamp', 'edited_timestamp', 'flags', 'components', 'id', 'channel_id', 'guild_id', 'pinned', 'mention_everyone', 'tts', 'author.id', 'author.username', 'author.avatar', 'author.discriminator', 'author.public_flags', 'author.flags', 'author.global_name')

Arquivos com erro de leitura: 0


### Etapa 3: verificar os tipos de dados



In [ ]:

import pyarrow.parquet as pq
from pathlib import Path
from collections import Counter

pasta = Path("/content/dataset_extraido")
arquivos = sorted(pasta.rglob("*.parquet"))

esquemas_tipos = Counter()
erros = []

for arquivo in arquivos:
    try:
        schema = pq.read_schema(arquivo)

        # Registra o nome e o tipo de cada coluna
        esquema = tuple(
            (campo.name, str(campo.type))
            for campo in schema
        )

        esquemas_tipos[esquema] += 1

    except Exception as erro:
        erros.append((arquivo.name, str(erro)))

print(f"Esquemas de tipos diferentes: {len(esquemas_tipos)}")

for i, (esquema, quantidade) in enumerate(esquemas_tipos.items(), start=1):
    print(f"\nEsquema {i}: {quantidade:,} arquivo(s)")

    for coluna, tipo in esquema:
        print(f"  {coluna}: {tipo}")

print(f"\nErros: {len(erros)}")

Esquemas de tipos diferentes: 1409

Esquema 1: 1 arquivo(s)
  type: int64
  content: large_string
  mentions: list<element: struct<id: string, username: string, avatar: string, discriminator: string, public_flags: int64, flags: int64, banner: null, accent_color: null, global_name: string, avatar_decoration_data: struct<asset: string, sku_id: string, expires_at: null>, collectibles: struct<nameplate: struct<sku_id: string, asset: string, label: string, palette: string>>, display_name_styles: null, banner_color: null, clan: struct<identity_guild_id: string, identity_enabled: bool, tag: string, badge: string>, primary_guild: struct<identity_guild_id: string, identity_enabled: bool, tag: string, badge: string>, bot: bool>>
  mention_roles: list<element: null>
  attachments: list<element: struct<id: string, filename: string, size: int64, url: string, proxy_url: string, width: int64, height: int64, content_type: string, content_scan_version: int64, placeholder: string, placeholder_version: i

### Etapa 4: Verificando valores ausentes

In [ ]:
pasta = Path("/content/dataset_extraido")
arquivos = sorted(pasta.rglob("*.parquet"))

nulos_por_coluna = None
total_linhas = 0

for i, arquivo in enumerate(arquivos, start=1):
    tabela = pq.read_table(arquivo)
    df_temp = tabela.to_pandas()

    nulos = df_temp.isna().sum()

    if nulos_por_coluna is None:
        nulos_por_coluna = nulos
    else:
        nulos_por_coluna = nulos_por_coluna.add(
            nulos, fill_value=0
        )

    total_linhas += len(df_temp)

    del tabela, df_temp

    if i % 250 == 0:
        print(f"Arquivos processados: {i:,}/{len(arquivos):,}")

resultado = pd.DataFrame({
    "valores_ausentes": nulos_por_coluna.astype("int64")
})

resultado["percentual_ausente"] = (
    resultado["valores_ausentes"] / total_linhas * 100
).round(2)

display(resultado.sort_values(
    "percentual_ausente",
    ascending=False
))

print(f"Total de mensagens analisadas: {total_linhas:,}")

Arquivos processados: 250/2,051
Arquivos processados: 500/2,051
Arquivos processados: 750/2,051
Arquivos processados: 1,000/2,051
Arquivos processados: 1,250/2,051
Arquivos processados: 1,500/2,051
Arquivos processados: 1,750/2,051
Arquivos processados: 2,000/2,051


,valores_ausentes,percentual_ausente
edited_timestamp,15653076,98.53
author.global_name,5761001,36.26
author.avatar,1607022,10.12
mention_roles,0,0.00
type,0,0.00
content,0,0.00
mentions,0,0.00
timestamp,0,0.00
embeds,0,0.00
attachments,0,0.00


Total de mensagens analisadas: 15,886,553


## Interpretação do resultado acima

Faz sentido que essas 3 colunas possuam muitos valores nulos, pois a maioria das mensagens não foi alterada, o avatar do autor não é importante para nós e o nome global pode não estar disponível. Portanto, os resultados seguem o esperado até então.

### Etapa 5: Validar Mensagens Duplicadas

Nesta etapa, verificamos a existência de IDs de mensagens duplicados em todo o dataset (inclusive entre arquivos diferentes).

In [ ]:
import pandas as pd
from pathlib import Path
import pyarrow.parquet as pq

pasta = Path("/content/dataset_extraido")
arquivos = sorted(pasta.rglob("*.parquet"))

# Para economizar memória com 15.8M de linhas, faremos a contagem usando conjuntos e dicionários rápidos
ids_vistos = set()
ids_duplicados = set()
total_ids_analisados = 0

for i, arquivo in enumerate(arquivos, start=1):
    # Lemos apenas a coluna de ID para otimizar tempo e memória
    tabela = pq.read_table(arquivo, columns=['id'])
    ids = tabela.column('id').to_pylist()

    for msg_id in ids:
        total_ids_analisados += 1
        if msg_id in ids_vistos:
            ids_duplicados.add(msg_id)
        else:
            ids_vistos.add(msg_id)

    if i % 500 == 0 or i == len(arquivos):
        print(f"Processados {i}/{len(arquivos)} arquivos...")

print(f"\nTotal de IDs analisados: {total_ids_analisados:,}")
print(f"Total de IDs únicos: {len(ids_vistos):,}")
print(f"Total de IDs duplicados encontrados: {len(ids_duplicados):,}")

Processados 500/2051 arquivos...
Processados 1000/2051 arquivos...
Processados 1500/2051 arquivos...
Processados 2000/2051 arquivos...
Processados 2051/2051 arquivos...

Total de IDs analisados: 15,886,553
Total de IDs únicos: 15,886,553
Total de IDs duplicados encontrados: 0


### Etapa 6: Validar IDs e Campos Obrigatórios

Verificação de IDs nulos ou vazios e se os campos obrigatórios básicos de consistência estão íntegros.

In [ ]:
import pyarrow.parquet as pq
from pathlib import Path

pasta = Path("/content/dataset_extraido")
arquivos = sorted(pasta.rglob("*.parquet"))

ids_invalidos = 0
autores_invalidos = 0

for i, arquivo in enumerate(arquivos, start=1):
    tabela = pq.read_table(arquivo, columns=['id', 'author.id'])
    df_temp = tabela.to_pandas()

    # IDs vazios, nulos ou que não são numéricos (comum em IDs do Discord)
    invalid_ids_mask = df_temp['id'].isna() | (df_temp['id'].astype(str).str.strip() == '')
    ids_invalidos += invalid_ids_mask.sum()

    # IDs do Autor vazios ou nulos
    invalid_author_mask = df_temp['author.id'].isna() | (df_temp['author.id'].astype(str).str.strip() == '')
    autores_invalidos += invalid_author_mask.sum()

print(f"Total de IDs de mensagens inválidos/vazios: {ids_invalidos}")
print(f"Total de IDs de autores inválidos/vazios: {autores_invalidos}")

### Etapa 7: Verificar Datas e Conteúdo

Procurar timestamps inválidos, conteúdos nulos e analisar se mensagens sem texto possuem anexo ou embed de mídia.

In [ ]:
import pyarrow.parquet as pq
import pandas as pd
from pathlib import Path

pasta = Path("/content/dataset_extraido")
arquivos = sorted(pasta.rglob("*.parquet"))

timestamps_invalidos = 0
mensagens_vazias_sem_midia = 0

for i, arquivo in enumerate(arquivos, start=1):
    tabela = pq.read_table(arquivo, columns=['timestamp', 'content', 'attachments', 'embeds'])
    df_temp = tabela.to_pandas()

    # 1. Verificar timestamps inválidos (tentativa de conversão)
    # Se já estiver em datetime, verificamos nulos. Caso contrário, forçamos conversão.
    if not pd.api.types.is_datetime64_any_dtype(df_temp['timestamp']):
        timestamps_invalidos += pd.to_datetime(df_temp['timestamp'], errors='coerce').isna().sum()
    else:
        timestamps_invalidos += df_temp['timestamp'].isna().sum()

    # 2. Mensagens sem texto e sem anexos/embeds
    # Uma mensagem é considerada textualmente vazia se for nula ou string em branco
    texto_vazio = df_temp['content'].isna() | (df_temp['content'].astype(str).str.strip() == '')

    # Verificar se as listas de attachments ou embeds estão vazias
    sem_attachments = df_temp['attachments'].apply(lambda x: len(x) == 0 if hasattr(x, '__len__') else True)
    sem_embeds = df_temp['embeds'].apply(lambda x: len(x) == 0 if hasattr(x, '__len__') else True)

    vazia_e_sem_midia = texto_vazio & sem_attachments & sem_embeds
    mensagens_vazias_sem_midia += vazia_e_sem_midia.sum()

print(f"Total de Timestamps inválidos/nulos: {timestamps_invalidos}")
print(f"Total de mensagens vazias e sem nenhum anexo/embed: {mensagens_vazias_sem_midia}")

### Etapa 8: Consolidar o Inventário dos Arquivos Parquet

Criação de um relatório detalhado com informações de cada arquivo individual.

In [ ]:
import os
import pyarrow.parquet as pq
import pandas as pd
from pathlib import Path

pasta = Path("/content/dataset_extraido")
arquivos = sorted(pasta.rglob("*.parquet"))

inventario_dados = []

for i, arquivo in enumerate(arquivos, start=1):
    tamanho_bytes = arquivo.stat().st_size
    tamanho_mb = tamanho_bytes / (1024 * 1024)

    try:
        metadata = pq.read_metadata(arquivo)
        num_linhas = metadata.num_rows
        possivel_problema = "Nenhum" if num_linhas > 0 else "Arquivo Vazio"
    except Exception as e:
        num_linhas = 0
        possivel_problema = f"Erro de leitura: {str(e)}"

    inventario_dados.append({
        "caminho_relativo": str(arquivo.relative_to(pasta)),
        "quantidade_mensagens": num_linhas,
        "tamanho_mb": round(tamanho_mb, 4),
        "status_problemas": possivel_problema
    })

    if i % 500 == 0 or i == len(arquivos):
        print(f"Catalogado {i}/{len(arquivos)} arquivos no inventário...")

df_inventario = pd.DataFrame(inventario_dados)
df_inventario.to_csv("/content/inventario_parquet.csv", index=False, encoding='utf-8')

print("\nInventário gerado e salvo com sucesso em '/content/inventario_parquet.csv'!")
display(df_inventario.head(20))

### Análise de Validação: Mensagens Respondidas e Anexos

Vamos rodar uma verificação em uma amostra dos arquivos (ou de forma agregada e otimizada) para entender a distribuição do campo `type` (que diferencia mensagens comuns de respostas no Discord) e dos campos de mídia.

In [5]:
import pyarrow.parquet as pq
import pandas as pd
from pathlib import Path
from collections import Counter

pasta = Path("/content/dataset_extraido")
arquivos = sorted(pasta.rglob("*.parquet"))

# Vamos analisar os primeiros 100 arquivos para entender a estrutura dos dados reais
tipos_mensagem = Counter()
mensagens_com_anexo = 0
mensagens_anexo_e_texto_vazio = 0
total_mensagens_amostra = 0

for i, arquivo in enumerate(arquivos[:100]):
    tabela = pq.read_table(arquivo, columns=['type', 'content', 'attachments'])
    df = tabela.to_pandas()

    total_mensagens_amostra += len(df)
    tipos_mensagem.update(df['type'].tolist())

    # Verificar attachments preenchidos
    tem_anexo = df['attachments'].apply(lambda x: len(x) > 0 if hasattr(x, '__len__') else False)
    mensagens_com_anexo += tem_anexo.sum()

    texto_vazio = df['content'].isna() | (df['content'].astype(str).str.strip() == '')
    mensagens_anexo_e_texto_vazio += (tem_anexo & texto_vazio).sum()

print(f"--- Análise em Amostra (Primeiros 100 arquivos: {total_mensagens_amostra:,} mensagens) ---")
print("\nDistribuição do campo 'type' (ID do tipo de mensagem no Discord):")
for t, qtd in tipos_mensagem.items():
    # No Discord: 0 = Default, 19 = Reply (Resposta)
    significado = "Normal (Default)" if t == 0 else ("Resposta (Reply)" if t == 19 else "Outro")
    print(f"  Tipo {t} ({significado}): {qtd:,} ocorrências")

print(f"\nMensagens com Anexo: {mensagens_com_anexo:,}")
print(f"Mensagens com Anexo e Sem Texto: {mensagens_anexo_e_texto_vazio:,}")


--- Análise em Amostra (Primeiros 100 arquivos: 47,771 mensagens) ---

Distribuição do campo 'type' (ID do tipo de mensagem no Discord):
  Tipo 20 (Outro): 1,205 ocorrências
  Tipo 0 (Normal (Default)): 35,949 ocorrências
  Tipo 19 (Resposta (Reply)): 10,567 ocorrências
  Tipo 6 (Outro): 10 ocorrências
  Tipo 24 (Outro): 2 ocorrências
  Tipo 18 (Outro): 7 ocorrências
  Tipo 7 (Outro): 27 ocorrências
  Tipo 46 (Outro): 3 ocorrências
  Tipo 12 (Outro): 1 ocorrências

Mensagens com Anexo: 1,945
Mensagens com Anexo e Sem Texto: 1,522


In [6]:
import os
from pathlib import Path

extract_path = Path('/content/dataset_extraido')

# Verificar se a pasta existe e listar o que tem dentro
if extract_path.exists():
    itens = list(extract_path.iterdir())
    print(f"A pasta {extract_path} existe e contém {len(itens)} itens imediatos.")
    print("Exemplos de itens encontrados:")
    for item in itens[:10]:
        print(f"  - {item.name} ({'Diretório' if item.is_dir() else 'Arquivo'})")
else:
    print(f"A pasta {extract_path} não existe no ambiente virtual!")


A pasta /content/dataset_extraido existe e contém 2052 itens imediatos.
Exemplos de itens encontrados:
  - galerinha_dos_gamers_chamada_de_voz_ativa___artes.parquet (Arquivo)
  - sintase_do_webnamoro_evento_agendado___animais.parquet (Arquivo)
  - 4_mencoes_anihub_777_larpvalley___pedir-musicas.parquet (Arquivo)
  - ominio___nformacoes.parquet (Arquivo)
  - 1_mencao_last_dream___saida.parquet (Arquivo)
  - 4_mencoes_iceberg_2k_evento_agendado___batepapo.parquet (Arquivo)
  - 1_mencao_sanatorio___dream-team.parquet (Arquivo)
  - 1_mencao_sanatorio___links.parquet (Arquivo)
  - god_galio___-comando.parquet (Arquivo)
  - district_of_the_red_thread___musica.parquet (Arquivo)


In [7]:
# Vamos fazer uma busca profunda para localizar onde os arquivos .parquet realmente estão salvos
todos_arquivos_parquet = list(Path('/content').rglob('*.parquet'))
print(f"Total de arquivos .parquet encontrados em todo o ambiente /content: {len(todos_arquivos_parquet)}")
if todos_arquivos_parquet:
    print("Caminho do primeiro arquivo encontrado:", todos_arquivos_parquet[0])


KeyboardInterrupt: 